# Boiling point 0 — Acquire NIST data

This provenance notebook is not run live in the workshop. It is the compact, current NistChemPy 2.0 workflow used to query the NIST Chemistry WebBook, collect phase-change records, and create a new table of canonical SMILES and boiling points.

Unlike the historical `CompoundData.parquet` snapshot, the output deliberately stores only `SMILES` and `BP`; it is the appropriate starting point for the subsequent graph-neural-network workflow.


In [8]:
from pathlib import Path
import re
import time

import nistchempy as nist
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from rdkit import Chem
from tqdm.auto import tqdm

OUTPUT_DIR = Path('outputs/nist_acquisition')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT = OUTPUT_DIR / 'NIST_BP_SMILES.parquet'
PROGRESS = OUTPUT_DIR / 'bp_completed_nist_ids.txt'

RUN_LIVE_COLLECTION = True
MAX_COMPOUNDS = None  # Set to None only for an intentional full crawl.
CHECKPOINT_EVERY = 25
request_config = nist.RequestConfig(delay=3.0, max_attempts=3, kwargs={'timeout': 30.0})

if nist.__version__ != '2.0.0':
    raise RuntimeError(f'Tested with nistchempy 2.0.0; found {nist.__version__}.')
print(f'NistChemPy {nist.__version__}; live collection: {RUN_LIVE_COLLECTION}')


NistChemPy 2.0.0; live collection: True


In [4]:
# Current NistChemPy 2.0 search: neutral compounds advertising phase-change data.
# The formula subdivisions avoid WebBook result truncation.
search_parameters = nist.NistSearchParameters(no_ion=True, allow_other=False, allow_extra=False, cTP=True)

def discover_compound_ids(max_carbons=19):
    identifiers = set()
    for carbon_count in tqdm(range(1, max_carbons + 1), desc='Discovering NIST records'):
        result = nist.run_search(f'C{carbon_count}H*O*N*S*Cl*', 'formula',
                                 search_parameters=search_parameters, request_config=request_config)
        if result.success:
            identifiers.update(result.compound_ids)
        if result.lost:
            for hydrogen_count in range(50):
                narrowed = nist.run_search(f'C{carbon_count}H{hydrogen_count}O*N*S*Cl*', 'formula',
                                           search_parameters=search_parameters, request_config=request_config)
                if narrowed.success:
                    identifiers.update(narrowed.compound_ids)
    return sorted(identifiers)

# Run this only as part of a deliberate new collection. Save the ID list for provenance.
if RUN_LIVE_COLLECTION:
    compound_ids = discover_compound_ids()
    pd.DataFrame({'NIST_molecule': compound_ids}).to_parquet(OUTPUT_DIR / 'bp_candidate_ids.parquet', index=False)
    print(f'Discovered {len(compound_ids):,} candidate records.')


Discovering NIST records: 100%|██████████| 19/19 [36:47<00:00, 116.20s/it]


Discovered 12,296 candidate records.


In [5]:
def extract_boiling_point(phase_change_url):
    """Return the first value explicitly labeled T_boil in the NIST phase-change table."""
    if not phase_change_url:
        return None
    response = requests.get(phase_change_url, timeout=30)
    response.raise_for_status()
    time.sleep(request_config.delay)
    for row in BeautifulSoup(response.text, 'html.parser').select('tr'):
        cells = row.find_all(['th', 'td'])
        if len(cells) < 2:
            continue
        label = re.sub(r'[^a-z]', '', cells[0].get_text(' ', strip=True).lower())
        if 'tboil' not in label and 'boiling' not in label:
            continue
        number = re.search(r'[-+]?\d*\.?\d+', cells[1].get_text(' ', strip=True))
        if number:
            return float(number.group())
    return None

def record_from_nist(compound_id):
    compound = nist.get_compound(compound_id, request_config=request_config)
    if compound is None:
        raise LookupError('No unique NIST record.')
    phase_change_url = compound.data_refs.get('cTP')
    boiling_point = extract_boiling_point(phase_change_url)
    if boiling_point is None:
        raise LookupError('No explicitly labeled boiling point.')
    molblock = compound.get_mol2D()
    molecule = Chem.MolFromMolBlock(molblock, sanitize=True, removeHs=True)
    if molecule is None:
        raise ValueError('NIST 2-D molfile could not be converted to RDKit.')
    return {'SMILES': Chem.MolToSmiles(molecule, canonical=True), 'BP': boiling_point}

# One-record API and schema test. This confirms the current NistChemPy 2 workflow.
example = record_from_nist('C106365')
print(example)


{'SMILES': 'CCCOC(=O)CC', 'BP': 396.0}


In [ ]:
if RUN_LIVE_COLLECTION:
    candidate_file = OUTPUT_DIR / 'bp_candidate_ids.parquet'
    if not candidate_file.exists():
        raise FileNotFoundError('Run discovery first or provide bp_candidate_ids.parquet.')
    candidate_ids = pd.read_parquet(candidate_file)['NIST_molecule'].tolist()
    if MAX_COMPOUNDS is not None:
        candidate_ids = candidate_ids[:MAX_COMPOUNDS]
    records = pd.read_parquet(OUTPUT).to_dict('records') if OUTPUT.exists() else []
    completed = set(PROGRESS.read_text(encoding='utf-8').splitlines()) if PROGRESS.exists() else set()
    failures = []
    for count, compound_id in enumerate(tqdm(candidate_ids, desc='Collecting BP and SMILES'), start=1):
        if compound_id in completed:
            continue
        try:
            records.append(record_from_nist(compound_id)); completed.add(compound_id)
        except Exception as error:
            failures.append({'NIST_molecule': compound_id, 'error': repr(error)})
        if count % CHECKPOINT_EVERY == 0:
            pd.DataFrame(records, columns=['SMILES', 'BP']).dropna().drop_duplicates('SMILES').to_parquet(OUTPUT, index=False)
            PROGRESS.write_text('\n'.join(sorted(completed)), encoding='utf-8')
    result = pd.DataFrame(records, columns=['SMILES', 'BP']).dropna().drop_duplicates('SMILES')
    result.to_parquet(OUTPUT, index=False)
    PROGRESS.write_text('\n'.join(sorted(completed)), encoding='utf-8')
    pd.DataFrame(failures).to_csv(OUTPUT_DIR / 'bp_failures.csv', index=False)
    print(f'Wrote {len(result):,} records to {OUTPUT}.')
else:
    print('Live collection disabled. Set RUN_LIVE_COLLECTION=True outside the workshop.')


[18:50:35] Warning: ambiguous stereochemistry - opposing bonds have opposite wedging - at atom 6 ignored.

## Handoff

`01_BP_1_Chemprop_GNN.ipynb` uses `outputs/nist_acquisition/NIST_BP_SMILES.parquet` when it exists. Otherwise it falls back to the packaged historical snapshot so the workshop demo remains reproducible and offline-friendly.
